# Stage A — B-variant probe: batch 4 / grad-accum 16 (recovery from OOM)

v1 failed at cell 9: `gpu_probe --config B --batch 8 --grad-accum 8` raised
`FileNotFoundError: /kaggle/working/probe-b.json` because the probe exited
before writing (peak activation memory at micro-batch 8 exceeded the T4's headroom).

Section 9 of the notebook says exactly this: halve --batch, double --grad-accum.
The learning update is unchanged: 4 x 1024 x 16 = 65,536 tokens per step,
identical to the 8 x 1024 x 8 that OOMed. Only the activation footprint drops,
because activations scale with the micro-batch, not grad-accum.

Same 5-step probe, same --json. If this one writes probe-b.json, cell 9 continues
to budget + cell 10 trains. If it still OOMs, the config does not fit this card
at ctx 1024 and the honest move is to halve block size (512) before retrying.


In [ ]:
!python -m training.scripts.gpu_probe --config B --tokenizer $TOKENIZER \
    --shards data/processed/stage_a/shards \
    --batch 4 --block 1024 --grad-accum 16 --steps 5 --amp \
    --json /kaggle/working/probe-b.json

import json, os
if not os.path.exists('/kaggle/working/probe-b.json'):
    raise SystemExit(
        'probe still wrote no file at batch 4 / grad-accum 16. '
        'Config B did not fit this card at ctx 1024: halve --block to 512 '
        'and re-probe before budgeting any steps.')
print(json.dumps(json.load(open('/kaggle/working/probe-b.json')), indent=2)[:1500])